# 02 — Feature Engineering

**Objectifs de cette étape :**
- Convertir la date en variables temporelles exploitables
- Encodage cyclique (sin/cos) pour heure et mois
- Création de lags (t-1, t-4, t-96) et moyennes glissantes (1h, 24h)
- Suppression des variables redondantes (CO2, Leading_Current_Power_Factor)
- Encodage des variables catégorielles (WeekStatus, Load_Type, Day_of_week)






.

In [10]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"C:/Users/kaism/OneDrive/Documents/Projet Perso/data/Steel_industry_data.csv")

# 1. Convertir la date
df['date'] = pd.to_datetime(df['date'], format='%d/%m/%Y %H:%M')

# 2. Extraire les variables temporelles
df['hour'] = df['date'].dt.hour
df['month'] = df['date'].dt.month
df['day_of_year'] = df['date'].dt.dayofyear

# 3. Encodage cyclique (sin/cos) pour heure et mois
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

# 4. Supprimer CO2 (corrélation 0.99 = data leakage)
df = df.drop(columns=['CO2(tCO2)'])

# 5. Supprimer Leading_Current_Power_Factor (redondant avec Lagging, corr -0.94)
df = df.drop(columns=['Leading_Current_Power_Factor'])

# 6. Encoder les variables catégorielles
df['WeekStatus'] = df['WeekStatus'].map({'Weekday': 1, 'Weekend': 0})
df['Load_Type'] = df['Load_Type'].map({'Light_Load': 0, 'Medium_Load': 1, 'Maximum_Load': 2})
df['Day_of_week'] = df['date'].dt.dayofweek  # 0=lundi, 6=dimanche

# 7. Vérification
print(df.shape)
print(df.dtypes)
df.head()

(35040, 16)
date                                    datetime64[us]
Usage_kWh                                      float64
Lagging_Current_Reactive.Power_kVarh           float64
Leading_Current_Reactive_Power_kVarh           float64
Lagging_Current_Power_Factor                   float64
NSM                                              int64
WeekStatus                                       int64
Day_of_week                                      int32
Load_Type                                        int64
hour                                             int32
month                                            int32
day_of_year                                      int32
hour_sin                                       float64
hour_cos                                       float64
month_sin                                      float64
month_cos                                      float64
dtype: object


,date,Usage_kWh,Lagging_Current_Reactive.Power_kVarh,Leading_Current_Reactive_Power_kVarh,Lagging_Current_Power_Factor,NSM,WeekStatus,Day_of_week,Load_Type,hour,month,day_of_year,hour_sin,hour_cos,month_sin,month_cos
0,2018-01-01 00:15:00,3.17,2.95,0.0,73.21,900,1,0,0,0,1,1,0.000000,1.000000,0.5,0.866025
1,2018-01-01 00:30:00,4.00,4.46,0.0,66.77,1800,1,0,0,0,1,1,0.000000,1.000000,0.5,0.866025
2,2018-01-01 00:45:00,3.24,3.28,0.0,70.28,2700,1,0,0,0,1,1,0.000000,1.000000,0.5,0.866025
3,2018-01-01 01:00:00,3.31,3.56,0.0,68.09,3600,1,0,0,1,1,1,0.258819,0.965926,0.5,0.866025
4,2018-01-01 01:15:00,3.82,4.50,0.0,64.72,4500,1,0,0,1,1,1,0.258819,0.965926,0.5,0.866025


In [2]:
# 8. Trier par date
df = df.sort_values('date').reset_index(drop=True)

# 9. Lags (consommation aux pas de temps précédents)
df['usage_lag_1'] = df['Usage_kWh'].shift(1)
df['usage_lag_4'] = df['Usage_kWh'].shift(4)    # 1h avant
df['usage_lag_96'] = df['Usage_kWh'].shift(96)  # 24h avant

# 10. Rolling means (moyennes glissantes)
df['usage_roll_4'] = df['Usage_kWh'].shift(1).rolling(window=4).mean()   # moyenne 1h
df['usage_roll_96'] = df['Usage_kWh'].shift(1).rolling(window=96).mean() # moyenne 24h

# 11. Supprimer les lignes avec NaN créées par les lags
df = df.dropna().reset_index(drop=True)

print(df.shape)
df.head()

(34944, 21)


,date,Usage_kWh,Lagging_Current_Reactive.Power_kVarh,Leading_Current_Reactive_Power_kVarh,Lagging_Current_Power_Factor,NSM,WeekStatus,Day_of_week,Load_Type,hour,...,day_of_year,hour_sin,hour_cos,month_sin,month_cos,usage_lag_1,usage_lag_4,usage_lag_96,usage_roll_4,usage_roll_96
0,2018-01-02 00:00:00,4.61,4.21,0.0,73.84,0,1,1,0,0,...,2,0.000000,1.000000,0.5,0.866025,3.67,3.53,3.42,3.4925,3.665208
1,2018-01-02 00:15:00,3.20,3.10,0.0,71.82,900,1,1,0,0,...,2,0.000000,1.000000,0.5,0.866025,4.61,3.53,3.17,3.7625,3.677604
2,2018-01-02 00:30:00,3.85,4.61,0.0,64.10,1800,1,1,0,0,...,2,0.000000,1.000000,0.5,0.866025,3.20,3.24,4.00,3.6800,3.677917
3,2018-01-02 00:45:00,3.28,3.67,0.0,66.64,2700,1,1,0,0,...,2,0.000000,1.000000,0.5,0.866025,3.85,3.67,3.24,3.8325,3.676354
4,2018-01-02 01:00:00,3.35,3.60,0.0,68.12,3600,1,1,0,1,...,2,0.258819,0.965926,0.5,0.866025,3.28,4.61,3.31,3.7350,3.676771


In [9]:
# 12. Sauvegarder le dataset transformé
df.to_csv("C:/Users/kaism/OneDrive/Documents/Projet Perso/data/steel_features.csv", index= False)

# 13. Définir X et y
X = df.drop(columns=['Usage_kWh', 'date'])
y = df['Usage_kWh']

# 14. Split temporel (80% train, 20% test) — PAS de random, on respecte l'ordre du temps
split_index = int(len(df) * 0.8)
X_train, X_test = X[:split_index], X[split_index:]
y_train, y_test = y[:split_index], y[split_index:]

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Train: (27955, 19), Test: (6989, 19)
